In [23]:
# One-time setup: install packages planned for the rest of the project.
# Run once, then restart the kernel (Restart button at the top of the notebook).
%pip install nbconvert scipy statsmodels seaborn tqdm
%pip install transformers sentence-transformers
%pip install lightgbm


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import json, random
import pandas as pd, numpy as np

LIMIT = None            # None = full file; set e.g. 500000 for a quick test
random.seed(0)
stats = {}              # parent_asin -> [n, sum, sumsq, n_half0, sum_half0, n_half1, sum_half1]
bad = 0

with open("Amazon/Office_Products.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if LIMIT and i >= LIMIT:
            break
        try:
            d = json.loads(line)
            a = d["parent_asin"]
            x = float(d["rating"])
        except Exception:
            bad += 1
            continue
        s = stats.get(a)
        if s is None:
            s = stats[a] = [0, 0.0, 0.0, 0, 0.0, 0, 0.0]
        s[0] += 1; s[1] += x; s[2] += x * x
        # random split of each product's reviews into two halves (for split-half reliability)
        if random.getrandbits(1):
            s[3] += 1; s[4] += x
        else:
            s[5] += 1; s[6] += x
        if i % 1_000_000 == 0:
            print(i, flush=True)

print(f"Read {i + 1} lines; bad lines: {bad}; products: {len(stats)}")

0
1000000
2000000
3000000
4000000
5000000
6000000
7000000
8000000
9000000
10000000
11000000
12000000
Read 12845712 lines; bad lines: 0; products: 710403


In [25]:
g = pd.DataFrame.from_dict(stats, orient="index",
        columns=["n", "sum", "sumsq", "n0", "s0", "n1", "s1"])
g.index.name = "parent_asin"
g = g.reset_index()
g["mean"] = g["sum"] / g["n"]
var = (g["sumsq"] - g["n"] * g["mean"] ** 2) / (g["n"] - 1)
g["sd"] = np.sqrt(var.clip(lower=0))
g["se"] = g["sd"] / np.sqrt(g["n"])
g["mean0"] = g["s0"] / g["n0"].replace(0, np.nan)
g["mean1"] = g["s1"] / g["n1"].replace(0, np.nan)
g.to_csv("Amazon/item_ratings.csv", index=False)   # cache so the big file is read only once

for t in [40, 100, 200]:
    print(f"products with >= {t} ratings:", (g.n >= t).sum())

products with >= 40 ratings: 50639
products with >= 100 ratings: 21462
products with >= 200 ratings: 9999


In [26]:
rows = []
with open("Amazon/meta_Office_Products.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        try:
            d = json.loads(line)
        except Exception:
            continue
        feats = d.get("features") or []
        desc = d.get("description") or []
        rows.append({
            "parent_asin": d.get("parent_asin"),
            "title": d.get("title"),
            "price": d.get("price"),
            "n_feat": len(feats),
            "desc_len": len(" ".join(desc)),
            "n_cat": len(d.get("categories") or []),
        })
m = pd.DataFrame(rows)
m["price"] = pd.to_numeric(m["price"], errors="coerce")
print(len(m), "metadata rows; share with a price:", round(m.price.notna().mean(), 3))

710503 metadata rows; share with a price: 0.397


In [27]:
df = g.merge(m, on="parent_asin", how="inner")
print("Metadata coverage of rated products:", round(len(df) / len(g), 3))
for t in [40, 100, 200]:
    s = df[df.n >= t]
    has_text = ((s.n_feat > 0) | (s.desc_len > 0)).mean()
    print(f"threshold {t}: products={len(s)}, with price={s.price.notna().mean():.2f}, with text={has_text:.2f}")

Metadata coverage of rated products: 1.0
threshold 40: products=50639, with price=0.70, with text=0.94
threshold 100: products=21462, with price=0.77, with text=0.96
threshold 200: products=9999, with price=0.81, with text=0.97


In [28]:
for t in [40, 100, 200]:
    s = g[g.n >= t].dropna(subset=["mean0", "mean1"])
    print(f"threshold {t}: split-half correlation of mean ratings =",
          round(s["mean0"].corr(s["mean1"]), 3))

threshold 40: split-half correlation of mean ratings = 0.873
threshold 100: split-half correlation of mean ratings = 0.935
threshold 200: split-half correlation of mean ratings = 0.963


In [29]:
# Audit: can we form functionally comparable, price-matched 4-product sets?
rows = []
with open("Amazon/meta_Office_Products.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        try:
            d = json.loads(line)
        except Exception:
            continue
        cats = d.get("categories") or []
        rows.append({
            "parent_asin": d.get("parent_asin"),
            "main_category": d.get("main_category"),
            "cat_depth": len(cats),
            "cat_leaf": cats[-1] if cats else None,
        })
c = pd.DataFrame(rows)

# Eligible pool: >=100 ratings, positive price, non-empty text fields
elig = df.merge(c, on="parent_asin", how="left")
elig = elig[(elig.n >= 100) & (elig.price > 0) & ((elig.n_feat > 0) | (elig.desc_len > 0))].copy()
print("Eligible products:", len(elig))

print("\nCategory depth (0 = category field empty):")
print(elig.cat_depth.value_counts().sort_index())

print("\nmain_category:")
print(elig.main_category.value_counts(dropna=False).head(10))

leaf = elig.groupby("cat_leaf").size().sort_values(ascending=False)
print("\nTop 20 leaf categories:")
print(leaf.head(20))
print("Leaf groups with >=4 products:", (leaf >= 4).sum(),
      "| products inside those groups:", leaf[leaf >= 4].sum())

# Upper bound on disjoint 4-sets with max/min price ratio <= 1.5, within each leaf category
def count_sets(prices, ratio=1.5, k=4):
    p = sorted(prices)
    i, sets = 0, 0
    while i + k <= len(p):
        if p[i + k - 1] / p[i] <= ratio:
            sets += 1
            i += k
        else:
            i += 1
    return sets

sets_per_leaf = elig.groupby("cat_leaf")["price"].apply(lambda s: count_sets(s.tolist()))
print("\nMax disjoint price-matched 4-sets (leaf level, upper bound):", int(sets_per_leaf.sum()))
print("Leaf categories contributing at least one set:", int((sets_per_leaf > 0).sum()))

# Eyeball the largest leaf category: are the products really the same kind of thing?
if len(leaf):
    top = leaf.index[0]
    print("\nSample titles from the largest leaf:", top)
    for t in elig[elig.cat_leaf == top].title.head(8):
        print(" -", t)

Eligible products: 16310

Category depth (0 = category field empty):
cat_depth
0     202
2     133
3     277
4    4496
5    8643
6    2559
Name: count, dtype: int64

main_category:
main_category
Office Products             11239
All Electronics              1675
Amazon Home                  1180
Tools & Home Improvement      537
Industrial & Scientific       449
Computers                     224
Arts, Crafts & Sewing         168
Toys & Games                  163
NaN                           157
Books                         149
Name: count, dtype: int64

Top 20 leaf categories:
cat_leaf
Inkjet Ink Cartridges            1192
Ballpoint Pens                    488
Toner Cartridges                  457
Greeting Cards                    443
Gel Ink Rollerball Pens           420
Badge Holders                     341
Mouse Pads                        323
Planners                          286
Dry Erase Boards                  240
Landline Phones                   215
Envelope Mailers         

In [30]:
# Audit 2: are rating differences WITHIN price-matched sets large and stable enough?
# Sets are built from category and price only; ratings are used afterwards, purely to audit the benchmark.
EXCLUDE = r"Cartridge|Toner|Printer|Greeting Card|Ink Refill"   # compatibility-bound or taste-driven; edit as needed

pool = elig[(elig.main_category == "Office Products")
            & elig.cat_leaf.notna()
            & ~elig.cat_leaf.str.contains(EXCLUDE, case=False, regex=True)].copy()
print("Pool after filters:", len(pool), "products;", pool.cat_leaf.nunique(), "leaf categories")

def make_sets(g_, ratio=1.5, k=4):
    g_ = g_.sort_values("price")
    idx, p = g_.index.tolist(), g_["price"].tolist()
    out, i = [], 0
    while i + k <= len(p):
        if p[i + k - 1] / p[i] <= ratio:
            out.append(idx[i:i + k]); i += k
        else:
            i += 1
    return out

sets = []
for leaf_name, g_ in pool.groupby("cat_leaf"):
    for s in make_sets(g_):
        sets.append((leaf_name, s))
print("Candidate sets (same leaf, price ratio <= 1.5):", len(sets))
print("Leaf categories with at least one set:", len({a for a, _ in sets}))

rec = []
for sid, (leaf_name, idxs) in enumerate(sets):
    sub = pool.loc[idxs]
    rec.append({
        "set_id": sid, "leaf": leaf_name,
        "range": sub["mean"].max() - sub["mean"].min(),
        "top_agree": sub["mean0"].values.argmax() == sub["mean1"].values.argmax(),
    })
S = pd.DataFrame(rec)

print("\nWithin-set rating range (stars):")
print(S["range"].describe(percentiles=[.1, .25, .5, .75, .9]).round(3))
print("Share of sets with range < 0.20:", round((S["range"] < 0.20).mean(), 3))
print("Top-1 split-half agreement (chance = 0.25):", round(S.top_agree.mean(), 3))

# Pooled within-set split-half correlation of mean ratings
W = pd.concat([pool.loc[idxs][["mean0", "mean1"]] - pool.loc[idxs][["mean0", "mean1"]].mean()
               for _, idxs in sets])
print("Within-set split-half correlation:", round(W["mean0"].corr(W["mean1"]), 3))

# Which leaf categories supply the most sets?
print("\nSets per leaf (top 15):")
print(S.groupby("leaf").size().sort_values(ascending=False).head(15))

Pool after filters: 9221 products; 387 leaf categories
Candidate sets (same leaf, price ratio <= 1.5): 1987
Leaf categories with at least one set: 211

Within-set rating range (stars):
count    1987.000
mean        0.733
std         0.458
min         0.040
10%         0.251
25%         0.394
50%         0.638
75%         0.961
90%         1.311
max         3.234
Name: range, dtype: float64
Share of sets with range < 0.20: 0.055
Top-1 split-half agreement (chance = 0.25): 0.664
Within-set split-half correlation: 0.902

Sets per leaf (top 15):
leaf
Ballpoint Pens                   113
Gel Ink Rollerball Pens           93
Badge Holders                     73
Planners                          69
Dry Erase Boards                  55
Envelope Mailers                  49
Hardcover Executive Notebooks     45
Mechanical Pencils                42
Fountain Pens                     41
Mouse Pads                        39
Self-Stick Notes                  38
Pen Refills                       36
All

In [31]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# (a) Is top-1 agreement driven by near-tie sets?
S["range_bin"] = pd.cut(S["range"], [0, 0.2, 0.4, 0.8, 5])
print("Top-1 split-half agreement by within-set range:")
print(S.groupby("range_bin", observed=True)["top_agree"].agg(["mean", "size"]).round(3))

# (b) Parse pack size from titles (unparsed -> 1) and keep only sets with identical pack size
PACK = re.compile(r"(?:pack of|set of|box of|case of)\s*(\d+)|(\d+)\s*[- ]?(?:pack|pk|count|ct|pcs|pieces|piece|pc|sets)\b", re.I)
def pack_count(title):
    if not isinstance(title, str):
        return 1
    m = PACK.search(title)
    if not m:
        return 1
    n = int(m.group(1) or m.group(2))
    return n if 1 <= n <= 1000 else 1

pool["pack"] = pool["title"].map(pack_count)
print("\nShare of pool with parsed pack size > 1:", round((pool.pack > 1).mean(), 3))

sets2 = []
for (leaf_name, pk), g_ in pool.groupby(["cat_leaf", "pack"]):
    for s in make_sets(g_):
        sets2.append((leaf_name, s))
print("Sets with identical pack size:", len(sets2), "| leaf categories:", len({a for a, _ in sets2}))
print(pd.Series([a for a, _ in sets2]).value_counts().head(10))

# (c) Title similarity within sets: a rough flag for functionally different products, not a substitute for manual review
tfidf = TfidfVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))
X = tfidf.fit_transform(pool["title"].fillna(""))
pos = {ix: k for k, ix in enumerate(pool.index)}

def mean_pair_cos(idxs):
    rows = [pos[i] for i in idxs]
    M = (X[rows] @ X[rows].T).toarray()
    return (M.sum() - np.trace(M)) / (len(rows) * (len(rows) - 1))

sim = pd.Series([mean_pair_cos(s) for _, s in sets2])
print("\nMean pairwise title cosine within sets:")
print(sim.describe(percentiles=[.1, .25, .5, .75]).round(3))

# Eyeball three random sets (titles, prices and pack sizes only; no ratings)
rng = np.random.default_rng(1)
for j in rng.choice(len(sets2), 3, replace=False):
    leaf_name, s = sets2[j]
    print("\nSet", j, "|", leaf_name)
    for i in s:
        print(f"  ${pool.loc[i, 'price']:.2f} | pack={pool.loc[i, 'pack']} | {pool.loc[i, 'title'][:90]}")

Top-1 split-half agreement by within-set range:
             mean  size
range_bin              
(0.0, 0.2]  0.200   110
(0.2, 0.4]  0.518   400
(0.4, 0.8]  0.705   780
(0.8, 5.0]  0.776   697

Share of pool with parsed pack size > 1: 0.338
Sets with identical pack size: 1500 | leaf categories: 179
Ballpoint Pens                   93
Gel Ink Rollerball Pens          77
Planners                         68
Badge Holders                    65
Dry Erase Boards                 47
Envelope Mailers                 43
Hardcover Executive Notebooks    43
Mouse Pads                       38
Fountain Pens                    35
Self-Stick Notes                 31
Name: count, dtype: int64

Mean pairwise title cosine within sets:
count    1500.000
mean        0.184
std         0.112
min         0.006
10%         0.061
25%         0.101
50%         0.159
75%         0.246
max         0.734
dtype: float64

Set 767 | Hardcover Executive Notebooks
  $16.99 | pack=1 | CAGIE A6 Refillable Faux Leather Tra

In [32]:
import re, torch
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

MASK_BRAND = False      # False = raw text; set True and re-run for a brand-masked version (closer to the 4435 materials)
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TAG = "masked" if MASK_BRAND else "raw"

def mask_store(text, store):
    s = (store or "").strip()
    return re.sub(re.escape(s), " ", text, flags=re.I) if len(s) >= 3 else text

texts = (T["title"] + ". " + T["features"] + " " + T["description"]).str.slice(0, 1500)
if MASK_BRAND:
    texts = pd.Series([mask_store(t, s) for t, s in zip(texts, T["store"])], index=T.index)

device = "mps" if torch.backends.mps.is_available() else "cpu"   # if mps raises an error, set device = "cpu"
print("device:", device)
st = SentenceTransformer(MODEL_NAME, device=device)
st.max_seq_length = 256
E = st.encode(texts.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
np.save(f"Amazon/emb_minilm_{TAG}.npy", E.astype("float32"))   # cached so this runs only once
print("Saved embeddings:", E.shape)

device: mps


Batches: 100%|██████████| 336/336 [03:34<00:00,  1.56it/s]


Saved embeddings: (21462, 384)


In [33]:
import json, re, hashlib
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

# Build the modeling table for products with >= 100 ratings, then freeze a family-grouped train/val/test split.
keep = set(g.loc[g.n >= 100, "parent_asin"])
rows = []
with open("Amazon/meta_Office_Products.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        try:
            d = json.loads(line)
        except Exception:
            continue
        a = d.get("parent_asin")
        if a not in keep:
            continue
        cats = d.get("categories") or []
        rows.append({
            "parent_asin": a,
            "title": d.get("title") or "",
            "features": " ".join(d.get("features") or []),
            "description": " ".join(d.get("description") or []),
            "price": d.get("price"),
            "store": d.get("store") or "",
            "main_category": d.get("main_category"),
            "cat_leaf": cats[-1] if cats else None,
        })
T = pd.DataFrame(rows)
T["price"] = pd.to_numeric(T["price"], errors="coerce")
T = T.merge(g[["parent_asin", "n", "mean", "sd", "se", "mean0", "mean1"]], on="parent_asin", how="left")
print("Products with >= 100 ratings:", len(T))

# Product families: near-duplicate titles within the same store (char 3-gram TF-IDF cosine >= 0.90)
T["title_norm"] = (T["title"].str.lower()
                   .str.replace(r"[^a-z0-9 ]+", " ", regex=True)
                   .str.replace(r"\s+", " ", regex=True).str.strip())
Xc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3)).fit_transform(T["title_norm"])

ei, ej = [], []
store_key = T["store"].str.lower().str.strip()
for _, idx in store_key.groupby(store_key).indices.items():
    if len(idx) < 2:
        continue
    for s in range(0, len(idx), 2000):
        sub = idx[s:s + 2000]
        M = (Xc[sub] @ Xc[idx].T).tocoo()
        r, c = sub[M.row], idx[M.col]
        keep_e = (M.data >= 0.90) & (r < c)
        ei.extend(r[keep_e].tolist()); ej.extend(c[keep_e].tolist())

N = len(T)
A = coo_matrix((np.ones(len(ei)), (np.array(ei, dtype=int), np.array(ej, dtype=int))), shape=(N, N))
n_fam, labels = connected_components(A, directed=False)
T["family"] = labels
fam_size = T["family"].map(T["family"].value_counts())
print("Families:", n_fam, "| products in multi-product families:", int((fam_size > 1).sum()),
      "| largest family:", int(fam_size.max()))

def assign(fam, seed=42):
    h = int(hashlib.md5(f"{seed}-{fam}".encode()).hexdigest(), 16) % 100
    return "train" if h < 60 else ("val" if h < 80 else "test")

T["split"] = T["family"].map(assign)
print(T["split"].value_counts().to_dict())

# Products that can enter price-matched sets (same rule as the earlier audit)
EXCLUDE = r"Cartridge|Toner|Printer|Greeting Card|Ink Refill"
T["set_eligible"] = ((T["main_category"] == "Office Products") & T["cat_leaf"].notna()
                     & ~T["cat_leaf"].fillna("").str.contains(EXCLUDE, case=False, regex=True)
                     & (T["price"] > 0) & ((T["features"].str.len() > 0) | (T["description"].str.len() > 0)))
print("Set-eligible products (earlier audit pool had 9221):", int(T["set_eligible"].sum()))
print(T[T["set_eligible"]].groupby("split").size().to_dict())

T.to_parquet("Amazon/model_table.parquet", index=False)
print("Saved Amazon/model_table.parquet")

Products with >= 100 ratings: 21462
Families: 20125 | products in multi-product families: 2283 | largest family: 9
{'train': 12934, 'val': 4337, 'test': 4191}
Set-eligible products (earlier audit pool had 9221): 9221
{'test': 1773, 'train': 5553, 'val': 1895}
Saved Amazon/model_table.parquet


In [34]:
import re, time
import numpy as np, pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---- Course-style model comparison, part 1: baselines, linear regression, ridge (GroupKFold CV).
# Only the train and validation splits are used here; the test split stays untouched.
T = pd.read_parquet("Amazon/model_table.parquet")

PACK = re.compile(r"(?:pack of|set of|box of|case of)\s*(\d+)|(\d+)\s*[- ]?(?:pack|pk|count|ct|pcs|pieces|piece|pc|sets)\b", re.I)
def pack_count(title):
    m = PACK.search(title) if isinstance(title, str) else None
    if not m:
        return 1
    n = int(m.group(1) or m.group(2))
    return n if 1 <= n <= 1000 else 1

def make_sets(g_, ratio=1.5, k=4):
    g_ = g_.sort_values("price")
    idx, p = g_.index.tolist(), g_["price"].tolist()
    out, i = [], 0
    while i + k <= len(p):
        if p[i + k - 1] / p[i] <= ratio:
            out.append(idx[i:i + k]); i += k
        else:
            i += 1
    return out

T["pack"] = T["title"].map(pack_count)
MANUAL_EXCLUDE = r"Executive Notebook|Pins & Tacks|Pen Refill|Planner"   # a priori judgment, made before looking at ratings
tr, va = T[T.split == "train"], T[T.split == "val"]
y_tr, y_va = tr["mean"].values, va["mean"].values
groups_tr = tr["family"].values
ybar = y_tr.mean()

def osr2(y, p, base):
    return 1 - ((y - p) ** 2).sum() / ((y - base) ** 2).sum()

# Features (all preprocessing is fitted on the training split only)
txt = lambda d: d["title"] + " " + d["features"] + " " + d["description"]
tfidf = TfidfVectorizer(min_df=3, ngram_range=(1, 2), max_features=50000, sublinear_tf=True).fit(txt(tr))
ohe = OneHotEncoder(handle_unknown="ignore").fit(tr[["cat_leaf"]].fillna("NA"))
med = tr["price"].median()
def num(d):
    return np.c_[np.log1p(d["price"].fillna(med)), d["price"].isna().astype(float),
                 np.log1p(d["title"].str.len()), np.log1p(d["features"].str.len()), np.log1p(d["description"].str.len())]
sc = StandardScaler().fit(num(tr))
def struct(d):
    return hstack([ohe.transform(d[["cat_leaf"]].fillna("NA")), csr_matrix(sc.transform(num(d)))]).tocsr()

Xs_tr, Xs_va = struct(tr), struct(va)
Xt_tr, Xt_va = tfidf.transform(txt(tr)), tfidf.transform(txt(va))
Xst_tr, Xst_va = hstack([Xs_tr, Xt_tr]).tocsr(), hstack([Xs_va, Xt_va]).tocsr()
E = np.load("Amazon/emb_minilm_raw.npy")
assert len(E) == len(T)
E_tr, E_va = E[tr.index.values], E[va.index.values]
XsE_tr, XsE_va = hstack([Xs_tr, csr_matrix(E_tr)]).tocsr(), hstack([Xs_va, csr_matrix(E_va)]).tocsr()

# Validation sets: built inside the val split only, from category, pack size and price (no ratings used)
ok = va[va["set_eligible"] & ~va["cat_leaf"].fillna("").str.contains(MANUAL_EXCLUDE, case=False, regex=True)]
sets_va = []
for (_, _), g_ in ok.groupby(["cat_leaf", "pack"]):
    sets_va += make_sets(g_)
pos = {ix: k for k, ix in enumerate(va.index)}
sets_pos = np.array([[pos[i] for i in s] for s in sets_va])
truth_sets = y_va[sets_pos]
R, O = truth_sets.mean(1), truth_sets.max(1)
print(f"Validation sets: {len(sets_va)} | headroom O-R: {np.mean(O - R):.3f} stars")

tie = np.random.default_rng(7).normal(0, 1e-9, len(va))   # fixed-seed tie breaking
def boot_ci(x, B=2000, seed=0):
    r = np.random.default_rng(seed); x = np.asarray(x)
    return np.percentile(x[r.integers(0, len(x), (B, len(x)))].mean(1), [5, 95])

results = []
def add_result(name, p_tr, p_va, tuned=""):
    j = (p_va + tie)[sets_pos].argmax(1)
    M = truth_sets[np.arange(len(j)), j]
    lo, hi = boot_ci(M - R)
    results.append({"model": name, "tuned": tuned,
                    "train R2": osr2(y_tr, p_tr, ybar), "val OSR2": osr2(y_va, p_va, ybar),
                    "val MAE": mean_absolute_error(y_va, p_va),
                    "val RMSE": mean_squared_error(y_va, p_va) ** 0.5,
                    "M-R": (M - R).mean(), "90% CI": f"[{lo:.3f}, {hi:.3f}]", "top1": (j == truth_sets.argmax(1)).mean()})

cv = GroupKFold(n_splits=3)
def tune_ridge(name, Xa, Xb):
    t0 = time.time()
    gs = GridSearchCV(Ridge(), {"alpha": [0.03, 0.1, 0.3, 1, 3, 10, 30, 100]}, cv=cv, scoring="neg_root_mean_squared_error")
    gs.fit(Xa, y_tr, groups=groups_tr)           # refit=True: best alpha is refit on the full training split
    add_result(name, gs.predict(Xa), gs.predict(Xb), f"alpha={gs.best_params_['alpha']}")
    print(f"{name}: done in {time.time() - t0:.0f}s")

# 1) Baselines
add_result("Baseline: train mean", np.full(len(tr), ybar), np.full(len(va), ybar))
lm = tr.groupby("cat_leaf")["mean"].mean()
add_result("Baseline: leaf mean", tr["cat_leaf"].map(lm).fillna(ybar).values, va["cat_leaf"].map(lm).fillna(ybar).values)

# 2) Linear regression and ridge (lambda chosen by GroupKFold CV on the training split)
ols = LinearRegression().fit(Xs_tr, y_tr)
add_result("Linear regression (structured)", ols.predict(Xs_tr), ols.predict(Xs_va))
tune_ridge("Ridge (structured)", Xs_tr, Xs_va)
tune_ridge("Ridge (structured + TF-IDF)", Xst_tr, Xst_va)
tune_ridge("Ridge (structured + MiniLM)", XsE_tr, XsE_va)

print(pd.DataFrame(results).round(3).to_string(index=False))

Validation sets: 148 | headroom O-R: 0.314 stars
Ridge (structured): done in 0s
Ridge (structured + TF-IDF): done in 29s
Ridge (structured + MiniLM): done in 16s
                         model   tuned  train R2  val OSR2  val MAE  val RMSE    M-R          90% CI  top1
          Baseline: train mean             0.000     0.000    0.394     0.502  0.027 [-0.014, 0.068] 0.257
           Baseline: leaf mean             0.297     0.274    0.322     0.428  0.027 [-0.014, 0.068] 0.257
Linear regression (structured)             0.303     0.278    0.321     0.426 -0.002 [-0.048, 0.041] 0.236
            Ridge (structured) alpha=3     0.292     0.286    0.321     0.424 -0.003 [-0.047, 0.039] 0.236
   Ridge (structured + TF-IDF) alpha=1     0.780     0.431    0.281     0.379  0.177  [0.140, 0.213] 0.473
   Ridge (structured + MiniLM) alpha=3     0.380     0.332    0.309     0.410  0.081  [0.041, 0.120] 0.311


In [35]:
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

# ---- Course-style model comparison, part 2: tree ensembles (GroupKFold CV on the training split).
# Trees get dense inputs: leaf-category dummies (frequent leaves only), standardized numeric features, and a text block.
svd = TruncatedSVD(n_components=100, random_state=0).fit(Xt_tr)
S_tr, S_va = svd.transform(Xt_tr), svd.transform(Xt_va)

freq = tr["cat_leaf"].value_counts()
leaves = freq[freq >= 50].index
def leaf_dummies(d):
    return pd.get_dummies(pd.Categorical(d["cat_leaf"], categories=leaves)).values.astype(float)

N_tr, N_va = sc.transform(num(tr)), sc.transform(num(va))
D_svd_tr, D_svd_va = np.hstack([leaf_dummies(tr), N_tr, S_tr]), np.hstack([leaf_dummies(va), N_va, S_va])
D_emb_tr, D_emb_va = np.hstack([leaf_dummies(tr), N_tr, E_tr]), np.hstack([leaf_dummies(va), N_va, E_va])
print("Tree inputs:", D_svd_tr.shape, D_emb_tr.shape)

def tune_tree(name, estimator, grid, Xa, Xb):
    t0 = time.time()
    gs = GridSearchCV(estimator, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    gs.fit(Xa, y_tr, groups=groups_tr)
    add_result(name, gs.predict(Xa), gs.predict(Xb), str(gs.best_params_))
    print(f"{name}: done in {time.time() - t0:.0f}s")

rf = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
tune_tree("Random forest (structured + TF-IDF SVD)", rf,
          {"max_features": [0.3, 0.6], "min_samples_leaf": [5, 20]}, D_svd_tr, D_svd_va)

hgb_grid = {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [8, 31], "max_iter": [100, 300]}
hgb = HistGradientBoostingRegressor(early_stopping=False, random_state=0)
tune_tree("Gradient boosting (structured + TF-IDF SVD)", hgb, hgb_grid, D_svd_tr, D_svd_va)
tune_tree("Gradient boosting (structured + MiniLM)", hgb, hgb_grid, D_emb_tr, D_emb_va)

print(pd.DataFrame(results).sort_values("val OSR2", ascending=False).round(3).to_string(index=False))

Tree inputs: (12934, 172) (12934, 456)


/var/folders/kc/fq_s4w916qdb5dzf9ltj10bh0000gn/T/ipykernel_9845/4248622936.py:12: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  return pd.get_dummies(pd.Categorical(d["cat_leaf"], categories=leaves)).values.astype(float)
/var/folders/kc/fq_s4w916qdb5dzf9ltj10bh0000gn/T/ipykernel_9845/4248622936.py:12: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  return pd.get_dummies(pd.Categorical(d["cat_leaf"], categories=leaves)).values.astype(float)
/var/folders/kc/fq_s4w916qdb5dzf9ltj10bh0000gn/T/ipykernel_9845/4248622936.py:12: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  return pd.get_dummies(pd.Categoric

Random forest (structured + TF-IDF SVD): done in 180s
Gradient boosting (structured + TF-IDF SVD): done in 59s
Gradient boosting (structured + MiniLM): done in 110s
                                      model                                                          tuned  train R2  val OSR2  val MAE  val RMSE    M-R          90% CI  top1
                Ridge (structured + TF-IDF)                                                        alpha=1     0.780     0.431    0.281     0.379  0.177  [0.140, 0.213] 0.473
Gradient boosting (structured + TF-IDF SVD) {'learning_rate': 0.05, 'max_iter': 300, 'max_leaf_nodes': 31}     0.729     0.411    0.287     0.385  0.155  [0.116, 0.199] 0.439
    Random forest (structured + TF-IDF SVD)                   {'max_features': 0.6, 'min_samples_leaf': 5}     0.806     0.406    0.287     0.387  0.132  [0.091, 0.176] 0.439
    Gradient boosting (structured + MiniLM)  {'learning_rate': 0.1, 'max_iter': 300, 'max_leaf_nodes': 31}     0.907     0.366    0.300